# Rung 1 — CAT-Q PTQ on C4

Sliding-layer reconstruction with learnable group modulation and softened ternarization. Calibration: 512 C4 samples x 2048 tokens.

## Setup

In [ ]:
# Colab Pro A100 setup. Persist results/checkpoints on Drive so a session
# expiry costs the run, not the artifacts.
from google.colab import drive
drive.mount("/content/drive")
REPO = "/content/drive/MyDrive/lfm3-ternary"
import os, subprocess, sys
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/ORG/LFM3-Ternary", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
os.chdir(REPO)
os.makedirs("results", exist_ok=True)
!pip -q install -r requirements.txt
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")


## Calibration corpus

In [ ]:
!python3 scripts/build_calib_c4.py --model LiquidAI/LFM3-1.7B \
    --out-dir data/calib --n-seq 512 --seq-len 2048 --seed 2

## Quantize

In [ ]:
# The rung runner writes the checkpoint and its ternary manifest; the eval
# script then produces the score file gates.py consumes.
!python3 scripts/run_catq.py --model LiquidAI/LFM3-1.7B --calib data/calib/c4_512x2048.pt --out checkpoints/rung1 --epochs 60
!bash scripts/eval_lm_eval.sh checkpoints/rung1/hf results/rung1.json
!python3 scripts/gates.py tier0 --baseline results/baseline.json --scores results/rung1.json \
    --rung 1 --remaining 2 --out results/rung1_verdict.json


## Inspect the verdict

In [ ]:
print(open("results/rung1_verdict.json").read())